# Excel Report Automation Tool
**Dataset:** Superstore Sales from Kaggle  
**Libraries:** pandas, openpyxl  

This is my first Python automation project. The idea is simple — take raw sales data and automatically generate a formatted Excel report with charts instead of doing it manually every time.

### Step 1 — Load and Clean the Data

Loading the CSV and fixing a few issues I found during exploration:
- Order Date and Ship Date were stored as plain text, not actual dates
- Postal Code was showing as 42420.0 (float) instead of 42420 (string)
- Also extracting year and month separately — need those for groupby later

In [ ]:
import pandas as pd

# need latin1 encoding or pandas throws a UnicodeDecodeError
df = pd.read_csv('train.csv', encoding='latin1')

# dates were stored as text, converting to datetime
# dayfirst=True because format is DD/MM/YYYY
df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=True)
df['Ship Date']  = pd.to_datetime(df['Ship Date'],  dayfirst=True)

# postal code was coming as 42420.0, fixing to string
df['Postal Code'] = df['Postal Code'].fillna(0).astype(int).astype(str)

# extracting year and month — will use for groupby
df['Order Year']       = df['Order Date'].dt.year
df['Order Month']      = df['Order Date'].dt.month
df['Order Month Name'] = df['Order Date'].dt.strftime('%b')

print('loaded! shape:', df.shape)
print('nulls remaining:\n', df.isnull().sum())

### Step 2 — Build Summary Tables

groupby works like a pivot table in Excel — groups rows by a column and runs an aggregation on another column.  
Built 3 tables: region-wise, category-wise, and month-wise.

In [ ]:
# total sales per region, sorted highest to lowest
region_sales = df.groupby('Region')['Sales'].sum().reset_index()
region_sales.columns = ['Region', 'Total Sales']
region_sales = region_sales.sort_values('Total Sales', ascending=False)

# total sales per category
category_sales = df.groupby('Category')['Sales'].sum().reset_index()
category_sales.columns = ['Category', 'Total Sales']
category_sales = category_sales.sort_values('Total Sales', ascending=False)

# month by month breakdown
monthly_sales = df.groupby(
    ['Order Year', 'Order Month', 'Order Month Name']
)['Sales'].sum().reset_index()
monthly_sales.columns = ['Year', 'Month Number', 'Month', 'Total Sales']
monthly_sales = monthly_sales.sort_values(['Year', 'Month Number'])

print('region sales:')
print(region_sales)
print('\ncategory sales:')
print(category_sales)

### Step 3 — Create the Excel Report

Using openpyxl to write everything into a formatted Excel file.  
Made two helper functions so I don't repeat the same styling code for every sheet:
- `style_header()` handles the colored header row
- `add_borders()` draws thin borders around all cells

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

wb = Workbook()
wb.remove(wb.active)  # excel adds a default blank sheet, don't need it

def style_header(ws, row, col_count, bg_color='2F5496'):
    for col in range(1, col_count + 1):
        cell = ws.cell(row=row, column=col)
        cell.font      = Font(bold=True, color='FFFFFF', size=11)
        cell.fill      = PatternFill('solid', fgColor=bg_color)
        cell.alignment = Alignment(horizontal='center')

def add_borders(ws, min_row, max_row, min_col, max_col):
    thin = Side(style='thin')
    for row in ws.iter_rows(min_row=min_row, max_row=max_row,
                             min_col=min_col, max_col=max_col):
        for cell in row:
            cell.border = Border(left=thin, right=thin,
                                  top=thin, bottom=thin)

# sheet 1 — sales by region
ws1 = wb.create_sheet('Sales by Region')
ws1['A1'] = 'Sales by Region'
ws1['A1'].font = Font(bold=True, size=14, color='2F5496')
ws1.merge_cells('A1:B1')
ws1['A2'] = 'Region'
ws1['B2'] = 'Total Sales ($)'
style_header(ws1, 2, 2)

for i, row in region_sales.iterrows():
    r = ws1.max_row + 1
    ws1.cell(r, 1, row['Region'])
    ws1.cell(r, 2, round(row['Total Sales'], 2))
    ws1.cell(r, 2).number_format = '#,##0.00'
    if r % 2 == 0:  # zebra stripes on even rows
        for c in [1, 2]:
            ws1.cell(r, c).fill = PatternFill('solid', fgColor='DCE6F1')

add_borders(ws1, 2, ws1.max_row, 1, 2)
ws1.column_dimensions['A'].width = 15
ws1.column_dimensions['B'].width = 20

# sheet 2 — sales by category
ws2 = wb.create_sheet('Sales by Category')
ws2['A1'] = 'Sales by Category'
ws2['A1'].font = Font(bold=True, size=14, color='2F5496')
ws2.merge_cells('A1:B1')
ws2['A2'] = 'Category'
ws2['B2'] = 'Total Sales ($)'
style_header(ws2, 2, 2, bg_color='375623')

for i, row in category_sales.iterrows():
    r = ws2.max_row + 1
    ws2.cell(r, 1, row['Category'])
    ws2.cell(r, 2, round(row['Total Sales'], 2))
    ws2.cell(r, 2).number_format = '#,##0.00'
    if r % 2 == 0:
        for c in [1, 2]:
            ws2.cell(r, c).fill = PatternFill('solid', fgColor='E2EFDA')

add_borders(ws2, 2, ws2.max_row, 1, 2)
ws2.column_dimensions['A'].width = 20
ws2.column_dimensions['B'].width = 20

# sheet 3 — monthly sales trend
ws3 = wb.create_sheet('Monthly Sales')
ws3['A1'] = 'Monthly Sales Trend'
ws3['A1'].font = Font(bold=True, size=14, color='2F5496')
ws3.merge_cells('A1:D1')
ws3['A2'] = 'Year'
ws3['B2'] = 'Month No.'
ws3['C2'] = 'Month'
ws3['D2'] = 'Total Sales ($)'
style_header(ws3, 2, 4, bg_color='7030A0')

for i, row in monthly_sales.iterrows():
    r = ws3.max_row + 1
    ws3.cell(r, 1, int(row['Year']))         # int() removes 2015.0 -> 2015
    ws3.cell(r, 2, int(row['Month Number']))
    ws3.cell(r, 3, row['Month'])
    ws3.cell(r, 4, round(row['Total Sales'], 2))
    ws3.cell(r, 4).number_format = '#,##0.00'
    if r % 2 == 0:
        for c in range(1, 5):
            ws3.cell(r, c).fill = PatternFill('solid', fgColor='EAD1F5')

add_borders(ws3, 2, ws3.max_row, 1, 4)
for col, width in zip(['A', 'B', 'C', 'D'], [10, 12, 12, 20]):
    ws3.column_dimensions[col].width = width

wb.save('Superstore_Sales_Report.xlsx')
print('saved!')

### Step 4 — Summary Cover Sheet

Added a summary tab that shows key numbers at a glance.  
`create_sheet('Summary', 0)` — the 0 puts it as the first tab.  
Used f-strings with `:,.2f` for formatting big numbers with commas.

In [ ]:
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

wb = load_workbook('Superstore_Sales_Report.xlsx')

total_sales      = df['Sales'].sum()
top_region       = region_sales.iloc[0]['Region']
top_region_sales = region_sales.iloc[0]['Total Sales']
top_category     = category_sales.iloc[0]['Category']
date_min         = df['Order Date'].dt.year.min()
date_max         = df['Order Date'].dt.year.max()
total_orders     = df['Order ID'].nunique()  # unique order IDs not row count

ws0 = wb.create_sheet('Summary', 0)  # 0 = insert at first position

ws0.column_dimensions['A'].width = 5
ws0.column_dimensions['B'].width = 28
ws0.column_dimensions['C'].width = 25

ws0.merge_cells('B2:C2')
ws0['B2'] = 'SUPERSTORE SALES REPORT'
ws0['B2'].font      = Font(bold=True, size=18, color='FFFFFF')
ws0['B2'].fill      = PatternFill('solid', fgColor='2F5496')
ws0['B2'].alignment = Alignment(horizontal='center', vertical='center')
ws0.row_dimensions[2].height = 35

ws0.merge_cells('B3:C3')
ws0['B3'] = f'Data Period: {date_min} to {date_max}'
ws0['B3'].font      = Font(italic=True, size=11, color='595959')
ws0['B3'].alignment = Alignment(horizontal='center')

def add_stat_row(ws, row, label, value):
    thin = Side(style='thin')
    ws.cell(row, 2, label).font = Font(bold=True, size=11)
    ws.cell(row, 2).fill        = PatternFill('solid', fgColor='DCE6F1')
    ws.cell(row, 2).alignment   = Alignment(vertical='center')
    ws.cell(row, 3, value).font = Font(size=11)
    ws.cell(row, 3).alignment   = Alignment(horizontal='left', vertical='center')
    for c in [2, 3]:
        ws.cell(row, c).border = Border(
            left=thin, right=thin, top=thin, bottom=thin)
    ws.row_dimensions[row].height = 22

ws0.merge_cells('B5:C5')
ws0['B5'] = 'KEY METRICS'
ws0['B5'].font      = Font(bold=True, size=12, color='FFFFFF')
ws0['B5'].fill      = PatternFill('solid', fgColor='2F5496')
ws0['B5'].alignment = Alignment(horizontal='center')

add_stat_row(ws0, 6,  'Total Revenue',  f'${total_sales:,.2f}')
add_stat_row(ws0, 7,  'Total Orders',   f'{total_orders:,}')
add_stat_row(ws0, 8,  'Top Region',     f'{top_region}  (${top_region_sales:,.2f})')
add_stat_row(ws0, 9,  'Top Category',   top_category)
add_stat_row(ws0, 10, 'Data Period',    f'{date_min} to {date_max}')

ws0.merge_cells('B12:C12')
ws0['B12'] = 'REPORT CONTENTS'
ws0['B12'].font      = Font(bold=True, size=12, color='FFFFFF')
ws0['B12'].fill      = PatternFill('solid', fgColor='2F5496')
ws0['B12'].alignment = Alignment(horizontal='center')

add_stat_row(ws0, 13, 'Sheet 1', 'Sales by Region')
add_stat_row(ws0, 14, 'Sheet 2', 'Sales by Category')
add_stat_row(ws0, 15, 'Sheet 3', 'Monthly Sales Trend')

ws0.merge_cells('B17:C17')
ws0['B17'] = 'Generated using Python  |  pandas + openpyxl'
ws0['B17'].font      = Font(italic=True, size=9, color='888888')
ws0['B17'].alignment = Alignment(horizontal='center')

wb.save('Superstore_Sales_Report.xlsx')
print('summary sheet done!')

### Step 5 — Add Charts

Embedding charts directly into the sheets using openpyxl's chart module.  
Reference() tells the chart which cells to pull data from.

In [ ]:
from openpyxl import load_workbook
from openpyxl.chart import BarChart, PieChart, LineChart, Reference

wb = load_workbook('Superstore_Sales_Report.xlsx')

# bar chart on region sheet
ws1 = wb['Sales by Region']
bar = BarChart()
bar.title        = 'Sales by Region'
bar.y_axis.title = 'Total Sales ($)'
bar.x_axis.title = 'Region'
bar.style        = 10
bar.width        = 15
bar.height       = 10
data = Reference(ws1, min_col=2, min_row=2, max_row=6)
cats = Reference(ws1, min_col=1, min_row=3, max_row=6)
bar.add_data(data, titles_from_data=True)
bar.set_categories(cats)
ws1.add_chart(bar, 'D2')

# pie chart on category sheet
ws2 = wb['Sales by Category']
pie = PieChart()
pie.title  = 'Sales by Category'
pie.style  = 10
pie.width  = 15
pie.height = 10
data = Reference(ws2, min_col=2, min_row=2, max_row=5)
cats = Reference(ws2, min_col=1, min_row=3, max_row=5)
pie.add_data(data, titles_from_data=True)
pie.set_categories(cats)
ws2.add_chart(pie, 'D2')

# line chart on monthly sales sheet
ws3   = wb['Monthly Sales']
line  = LineChart()
line.title        = 'Monthly Sales Trend'
line.y_axis.title = 'Total Sales ($)'
line.x_axis.title = 'Month'
line.style        = 10
line.width        = 25
line.height       = 12
max_r = ws3.max_row
data  = Reference(ws3, min_col=4, min_row=2, max_row=max_r)
cats  = Reference(ws3, min_col=3, min_row=3, max_row=max_r)
line.add_data(data, titles_from_data=True)
line.set_categories(cats)
ws3.add_chart(line, 'F2')

wb.save('Superstore_Sales_Report.xlsx')
print('charts added!')

### Step 6 — Download

In [ ]:
from google.colab import files
files.download('Superstore_Sales_Report.xlsx')